# U-JEPA Phase 0: LatentMAS baseline on Qwen3-14B

Gate: GSM8K accuracy >= 65 percent on 250 problems. Output goes to `/kaggle/working/results/phase0_baseline.json`. Requires GPU T4 x2 and Internet On.

**Note on disk**: HF model cache goes to `/tmp/hf_cache` (50+ GB ephemeral) because `/kaggle/working` is capped at 20 GB and Qwen3-14B is ~28 GB.

In [ ]:
# Cell 0: remove any old HF cache stuck inside /kaggle/working quota from a previous run
import shutil, os
stale = '/kaggle/working/hf_cache'
if os.path.isdir(stale):
    print(f'removing stale cache at {stale}')
    shutil.rmtree(stale)
os.makedirs('/tmp/hf_cache', exist_ok=True)
print('df -h /kaggle/working /tmp:')
import subprocess; print(subprocess.check_output(['df', '-h', '/kaggle/working', '/tmp']).decode())

In [ ]:
import subprocess, os, sys
if not os.path.exists('/kaggle/working/U-JEPA'):
    subprocess.run(['git', 'clone', 'https://github.com/kartikshirode/U-JEPA.git',
                    '/kaggle/working/U-JEPA'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/U-JEPA', 'pull'], check=True)
os.chdir('/kaggle/working/U-JEPA')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print(f'Warning: no HF_TOKEN secret found ({e}). Qwen3-14B is ungated so this is OK for Phase 0.')
# Force HF cache outside /kaggle/working quota
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['HF_HUB_CACHE'] = '/tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_cache'
print(f'HF_HOME={os.environ["HF_HOME"]}')

In [ ]:
import torch
print(f'torch {torch.__version__}, cuda {torch.version.cuda}, GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, '
          f'{torch.cuda.get_device_properties(i).total_memory // (1024**2)} MiB')

In [ ]:
import subprocess, sys, os
# Propagate cache env vars to the subprocess
env = os.environ.copy()
env['HF_HOME'] = '/tmp/hf_cache'
env['HF_HUB_CACHE'] = '/tmp/hf_cache'
env['TRANSFORMERS_CACHE'] = '/tmp/hf_cache'
subprocess.run([sys.executable, 'scripts/01_repro_latentmas_gsm8k.py'], check=True, env=env)

In [ ]:
import json
from pathlib import Path
p = Path('/kaggle/working/results/phase0_baseline.json')
if p.exists():
    print(json.dumps(json.loads(p.read_text()), indent=2))
else:
    print('No results file yet')
log = Path('/kaggle/working/results/phase0_baseline.log')
if log.exists():
    print('=== last 60 log lines ===')
    for line in log.read_text().splitlines()[-60:]:
        print(line)